In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from databricks.sdk.runtime import dbutils

In [ ]:
repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd())))
config_dir = os.path.join(repo_root, "config", "data_ingest")

with open(os.path.join(config_dir, "catalog_config.json")) as f:
    catalog = json.load(f)["catalog"]

daily_batch_config_path = os.path.join(config_dir, "daily_batch_config.json")
with open(daily_batch_config_path) as f:
    batch_config = json.load(f)

triggers = batch_config["triggers"]

In [ ]:
def run_worker(trigger):
    result = dbutils.notebook.run(
        "./worker",
        600,
        {
            "ticker": trigger["ticker"],
            "start_date": trigger["start_date"] or "",
            "catalog": catalog,
        },
    )
    return trigger["ticker"], result

results = {}
with ThreadPoolExecutor(max_workers=min(len(triggers), 8)) as executor:
    futures = {executor.submit(run_worker, trigger): trigger for trigger in triggers}
    for future in as_completed(futures):
        trigger = futures[future]
        try:
            ticker, exit_code = future.result()
            results[ticker] = exit_code
        except Exception as e:
            print(f"{trigger['ticker']} failed: {e}")
            results[trigger["ticker"]] = "1"

In [ ]:
for trigger in triggers:
    if results.get(trigger["ticker"]) != "0":
        continue
    table = trigger["ticker"].replace(".", "_")
    max_date = spark.sql(
        f"SELECT MAX(date) AS max_date FROM {catalog}.yfinance.{table}"
    ).collect()[0]["max_date"]
    trigger["start_date"] = str(max_date)

In [ ]:
with open(daily_batch_config_path, "w") as f:
    json.dump(batch_config, f, indent=2)